# Study 1: controlled paired-image versus constant-image QLoRA pilot

This launcher trains two otherwise identical diagnostic adapters on the leakage-safe training partition. The only experimental difference is whether each question receives its correct GI image or one shared neutral image. Each arm saves at step 128 and explicitly resumes to step 256.

Use Colab runtime version **2025.07**, select a **T4 GPU**, expose the read-capable `HF_TOKEN` secret, paste the full pushed Git commit below, and run all cells. Keep `RESET_RUN = False` when rerunning after an interrupted phase so complete checkpoints can be reused. This bounded pilot is excluded from paper results.

In [ ]:
from pathlib import Path
import re

REPOSITORY_URL = "https://github.com/dizza01/VLM.git"
REPOSITORY_COMMIT = "PASTE_FULL_40_CHARACTER_COMMIT_SHA_HERE"
CHECKOUT_ROOT = Path("/content/vlm-controlled-training")
INSTALL_SOURCE = Path("/content/gi-vqa-controlled-training-install")
WORK_DIR = Path("/content/gi_vqa_controlled_training_work")
ARTIFACT_DIR = Path("/content/gi_vqa_controlled_training_artifacts")
RESET_RUN = False
DOWNLOAD_BUNDLE = True

if re.fullmatch(r"[0-9a-fA-F]{40}", REPOSITORY_COMMIT) is None:
    raise ValueError("REPOSITORY_COMMIT must be the full 40-character SHA of the pushed commit")
REPOSITORY_COMMIT = REPOSITORY_COMMIT.lower()
PROJECT_ROOT = CHECKOUT_ROOT / "gi_vqa_research"
PROTOCOL_PATH = PROJECT_ROOT / "protocols/study1/controlled_training_pilot.json"
SPLIT_MANIFEST = PROJECT_ROOT / "protocols/study1/grouped_split_manifest.json"
print(f"Controlled-training commit: {REPOSITORY_COMMIT}")

In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess
import sys
import traceback

def run_command(command, *, cwd=None, capture=False):
    return subprocess.run(
        [str(part) for part in command], cwd=cwd, check=True,
        text=True, capture_output=capture,
    )

for path in (CHECKOUT_ROOT, INSTALL_SOURCE):
    if path.exists():
        shutil.rmtree(path)
if RESET_RUN:
    for path in (WORK_DIR, ARTIFACT_DIR):
        if path.exists():
            shutil.rmtree(path)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_OK = False

try:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError(
            "Python 3.11 is required. Select Colab runtime version 2025.07, reconnect, "
            f"and run from the top. Observed: {sys.version}"
        )
    gpu_query = run_command(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
        capture=True,
    ).stdout.strip().splitlines()
    if len(gpu_query) != 1 or "T4" not in gpu_query[0]:
        raise RuntimeError(f"Exactly one T4 GPU is required; observed {gpu_query}")

    run_command(["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, CHECKOUT_ROOT])
    run_command(["git", "checkout", "--detach", REPOSITORY_COMMIT], cwd=CHECKOUT_ROOT)
    resolved_commit = run_command(["git", "rev-parse", "HEAD"], cwd=CHECKOUT_ROOT, capture=True).stdout.strip().lower()
    if resolved_commit != REPOSITORY_COMMIT:
        raise RuntimeError(f"Resolved commit {resolved_commit} does not match {REPOSITORY_COMMIT}")
    for required_path in (PROJECT_ROOT, PROTOCOL_PATH, SPLIT_MANIFEST):
        if not required_path.exists():
            raise RuntimeError(f"Checked-out commit is missing: {required_path}")

    shutil.copytree(PROJECT_ROOT, INSTALL_SOURCE)
    run_command([sys.executable, "-m", "pip", "install", "--no-cache-dir", f"{INSTALL_SOURCE}[gpu]"])
    run_command([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps", str(INSTALL_SOURCE)])
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], check=False, capture_output=True, text=True,
    )
    pip_check_text = "\n".join(part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip())
    (ARTIFACT_DIR / "pip_check.txt").write_text((pip_check_text + "\n") if pip_check_text else "No broken requirements found.\n", encoding="utf-8")
    scoped = {"accelerate", "bitsandbytes", "datasets", "gi-vqa-research", "huggingface-hub", "ms-swift", "numpy", "peft", "pillow", "pyyaml", "sentencepiece", "torch", "transformers", "wandb"}
    scoped_conflicts = [line for line in pip_check_text.splitlines() if line and line.split(maxsplit=1)[0].lower().replace("_", "-") in scoped]
    if scoped_conflicts:
        raise RuntimeError("Project-stack dependency conflicts: " + " | ".join(scoped_conflicts))
    if pip_check.returncode != 0:
        print("Global pip check WARNING (unrelated Colab packages):")
        print(pip_check_text)

    installed_path = Path(run_command([sys.executable, "-c", "import gi_vqa.controlled_training_runner as m; print(m.__file__)"], capture=True).stdout.strip())
    source_path = PROJECT_ROOT / "src/gi_vqa/controlled_training_runner.py"
    installed_sha = hashlib.sha256(installed_path.read_bytes()).hexdigest()
    source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
    if installed_sha != source_sha:
        raise RuntimeError("Installed controlled-training runner differs from the verified checkout")
    git_status = run_command(["git", "status", "--porcelain=v1", "--untracked-files=all"], cwd=CHECKOUT_ROOT, capture=True).stdout.strip()
    if git_status:
        raise RuntimeError(f"Verified checkout became dirty: {git_status.splitlines()}")

    bootstrap = {"status": "PASS", "python": sys.version, "implementation": platform.python_implementation(), "gpu_query": gpu_query, "repository_url": REPOSITORY_URL, "repository_commit": resolved_commit, "checkout_clean": True, "installed_runner_path": str(installed_path), "installed_runner_sha256": installed_sha, "global_pip_check_exit_code": pip_check.returncode, "scoped_pip_conflicts": scoped_conflicts}
    (ARTIFACT_DIR / "bootstrap_environment.json").write_text(json.dumps(bootstrap, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    BOOTSTRAP_OK = True
    print("Bootstrap PASS: exact clean checkout and pinned GPU stack are ready.")
except BaseException as exc:
    failure = {"status": "FAIL", "phase": "bootstrap", "type": f"{type(exc).__module__}.{type(exc).__name__}", "message": re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", str(exc)), "traceback": re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", traceback.format_exc())}
    (ARTIFACT_DIR / "bootstrap_failure.json").write_text(json.dumps(failure, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"Bootstrap FAIL: {failure['message']}")

In [ ]:
import os

os.environ["HF_HOME"] = "/content/huggingface-cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"
AUTH_OK = False

if not BOOTSTRAP_OK:
    print("Authentication SKIPPED because bootstrap failed.")
else:
    try:
        from google.colab import userdata
        from huggingface_hub import login, whoami
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError("Colab secret HF_TOKEN is missing or notebook access is disabled")
        login(token=hf_token, add_to_git_credential=False)
        identity = whoami(token=hf_token)
        os.environ["HF_TOKEN"] = hf_token
        del hf_token
        AUTH_OK = True
        print(f"Hugging Face authentication PASS for user: {identity.get('name') or identity.get('fullname')}")
    except BaseException as exc:
        message = re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", str(exc))
        (ARTIFACT_DIR / "authentication_failure.json").write_text(json.dumps({"status": "FAIL", "phase": "authentication", "type": f"{type(exc).__module__}.{type(exc).__name__}", "message": message}, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        print(f"Authentication FAIL: {message}")

In [ ]:
TRAINING_OK = False
TRAINING_PROCESS = None
try:
    if not BOOTSTRAP_OK or not AUTH_OK:
        raise RuntimeError("Controlled training skipped because bootstrap or authentication failed")
    command = [
        sys.executable, "-m", "gi_vqa.controlled_training_runner",
        "--repository-root", str(CHECKOUT_ROOT),
        "--expected-commit", REPOSITORY_COMMIT,
        "--protocol", str(PROTOCOL_PATH),
        "--split-manifest", str(SPLIT_MANIFEST),
        "--data-dir", "data/controlled_training_pilot",
        "--work-dir", str(WORK_DIR),
        "--artifact-dir", str(ARTIFACT_DIR),
        "--required-gpu-substring", "T4",
    ]
    TRAINING_PROCESS = subprocess.run(command, check=False)
    report_path = ARTIFACT_DIR / "controlled_training_report.json"
    if not report_path.is_file():
        raise RuntimeError("Runner produced no controlled_training_report.json")
    TRAINING_REPORT = json.loads(report_path.read_text(encoding="utf-8"))
    if TRAINING_PROCESS.returncode != 0 or TRAINING_REPORT.get("status") != "PASS":
        raise RuntimeError(TRAINING_REPORT.get("error", {}).get("message", f"runner exit {TRAINING_PROCESS.returncode}"))
    TRAINING_OK = True
    print("Controlled training PASS")
    for condition, result in TRAINING_REPORT["conditions"].items():
        print(f"{condition}: 128 reused={result['phase_1']['reused']}, 256 reused={result['phase_2']['reused']}, reload loss={result['adapter_reload']['loss']:.7g}")
except BaseException as exc:
    print(f"Controlled training FAIL: {exc}")

In [ ]:
import zipfile

os.environ.pop("HF_TOKEN", None)
pip_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze", "--all"], check=True, capture_output=True, text=True).stdout
(ARTIFACT_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")
bundle_files = [path for path in sorted(ARTIFACT_DIR.rglob("*")) if path.is_file() and path.name not in {"controlled_training_bundle.zip", "bundle_manifest.json"}]
member_hashes = {str(path.relative_to(ARTIFACT_DIR)): hashlib.sha256(path.read_bytes()).hexdigest() for path in bundle_files}
bundle_manifest = {"controlled_training_status": "PASS" if TRAINING_OK else "FAIL", "repository_commit": REPOSITORY_COMMIT, "members_sha256": member_hashes}
manifest_path = ARTIFACT_DIR / "bundle_manifest.json"
manifest_path.write_text(json.dumps(bundle_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
BUNDLE_PATH = ARTIFACT_DIR / "controlled_training_bundle.zip"
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_STORED) as archive:
    for path in bundle_files:
        archive.write(path, arcname=str(path.relative_to(ARTIFACT_DIR)))
    archive.write(manifest_path, arcname="bundle_manifest.json")
BUNDLE_SHA256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
print(f"Evidence and adapter bundle: {BUNDLE_PATH}")
print(f"Bundle SHA-256: {BUNDLE_SHA256}")
if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(BUNDLE_PATH))

In [ ]:
if not TRAINING_OK:
    raise RuntimeError("Controlled training FAILED. Keep WORK_DIR and rerun with RESET_RUN=False; complete phase checkpoints will be reused.")
print("Controlled training PASS. Preserve the bundle with its Git commit. Next gate: compare the base, paired adapter and constant-image adapter on the locked development set.")